# Perishable-goods VQR training in Google Colab

This notebook trains small Variational Quantum Regressor candidates for `units_sold`, selects them using validation data, and evaluates the winner once on held-out test data. **No cell has been executed in the repository version.**

## 1. Configuration

`TRAIN_SAMPLE_SIZE = 100` is appropriate for checking the pipeline, not for a production-quality conclusion. Keep `FAST_MODE = True` for the first run. Increasing feature count increases the simulated qubit count and can make statevector training dramatically more expensive.

In [ ]:
REPOSITORY_URL = "<YOUR_REPOSITORY_URL>"
REPOSITORY_DIR = "/content/project"

USE_GOOGLE_DRIVE = False
GOOGLE_DRIVE_PROJECT_DIR = "/content/drive/MyDrive/fourstack-quantum-logistics"

FAST_MODE = True
MAXITER_FAST = 20
MAXITER_FINAL = 100
FEATURE_COUNTS = [4, 6, 8, 10, 12, 14]

TRAIN_SAMPLE_SIZE = 100
VALIDATION_SAMPLE_SIZE = 30
TEST_SAMPLE_SIZE = 30
RANDOM_STATE = 42

# Leave False unless spoilage_risk is known before prediction and is not
# derived from future sales, waste, or spoilage outcomes.
INCLUDE_SPOILAGE_RISK = False

EXPERIMENT_CONFIGS = [
    {
        "feature_map": "zz",
        "feature_map_reps": 1,
        "ansatz": "real_amplitudes",
        "ansatz_reps": 1,
        "entanglement": "linear",
        "optimizer": "COBYLA",
        "maxiter": 20,
    },
    {
        "feature_map": "z",
        "feature_map_reps": 1,
        "ansatz": "real_amplitudes",
        "ansatz_reps": 1,
        "entanglement": "linear",
        "optimizer": "COBYLA",
        "maxiter": 20,
    },
]

## 2. Colab environment setup

Run the dependency cell once in Colab. These versions come from `ml/uv.lock`; the cell does not blindly upgrade to unrelated latest versions.

In [ ]:
# Run this cell once in a fresh Colab runtime, then restart the runtime if Colab asks.
%pip install qiskit==2.5.0 qiskit-machine-learning==0.9.0 qiskit-algorithms==0.4.0 numpy==2.5.1 pandas==3.0.3 scikit-learn==1.9.0 scipy==1.18.0 joblib==1.5.3 dill==0.4.1 matplotlib==3.11.0

In [ ]:
from pathlib import Path
import subprocess

# Option A: clone a repository only after replacing the placeholder URL.
if REPOSITORY_URL != "<YOUR_REPOSITORY_URL>":
    repository_directory = Path(REPOSITORY_DIR)
    if not repository_directory.exists():
        repository_directory.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(
            ["git", "clone", REPOSITORY_URL, str(repository_directory)],
            check=True,
        )
    else:
        print(f"Clone skipped because {repository_directory} already exists.")
else:
    print("Clone skipped: replace <YOUR_REPOSITORY_URL> to use Option A.")

# Option B: mount Drive only when explicitly enabled.
if USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    print(f"Configured Drive project path: {GOOGLE_DRIVE_PROJECT_DIR}")

## 3. Imports and versions

In [ ]:
import gc
import importlib.metadata
import sys

import joblib
import matplotlib
import numpy as np
import pandas as pd
import qiskit
import sklearn

for package_name in [
    "qiskit",
    "qiskit-machine-learning",
    "qiskit-algorithms",
    "numpy",
    "pandas",
    "scikit-learn",
    "scipy",
    "joblib",
    "dill",
    "matplotlib",
]:
    print(f"{package_name}: {importlib.metadata.version(package_name)}")

## 4. Repository and path setup

The next cell detects a repository root by looking for `ml/algorithms/vqr.py`, adds that root to `sys.path`, and reuses shared constants from `ml/utils/path.py`.

In [ ]:
def is_repository_root(path: Path) -> bool:
    return (path / "ml" / "algorithms" / "vqr.py").is_file()


search_candidates = [
    Path.cwd(),
    *Path.cwd().parents,
    Path(REPOSITORY_DIR),
    Path(GOOGLE_DRIVE_PROJECT_DIR),
]
repository_root = next(
    (candidate.resolve() for candidate in search_candidates if is_repository_root(candidate)),
    None,
)
if repository_root is None:
    raise FileNotFoundError(
        "Repository root not found. Configure Option A or Option B, then rerun this cell."
    )
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

from ml.training.train_vqr_colab import (
    ExperimentConfig,
    WorkflowConfig,
    WorkflowPaths,
    build_prepared_data,
    effective_experiment_configs,
    fit_candidate_schema,
    fit_winner_and_evaluate_test,
    load_dataset,
    plot_actual_vs_predicted,
    plot_loss_curve,
    plot_training_duration,
    plot_validation_metric,
    prepare_raw_inputs,
    rank_successful_vqr,
    reload_and_test_inference,
    review_target_leakage,
    run_classical_baselines,
    run_vqr_experiments,
    save_experiment_tables,
    save_winning_artifacts,
    split_raw_data,
    validate_dataset,
)
from ml.utils.path import (
    PERISHABLE_GOODS_DATA_PATH,
    PERISHABLE_VQR_MODEL_PATH,
    VQR_EXPERIMENT_RESULTS_PATH,
    VQR_LOSS_HISTORY_PATH,
)

workflow_config = WorkflowConfig(
    fast_mode=FAST_MODE,
    maxiter_fast=MAXITER_FAST,
    maxiter_final=MAXITER_FINAL,
    feature_counts=tuple(FEATURE_COUNTS),
    train_sample_size=TRAIN_SAMPLE_SIZE,
    validation_sample_size=VALIDATION_SAMPLE_SIZE,
    test_sample_size=TEST_SAMPLE_SIZE,
    random_state=RANDOM_STATE,
    include_spoilage_risk=INCLUDE_SPOILAGE_RISK,
    experiment_configs=tuple(ExperimentConfig(**item) for item in EXPERIMENT_CONFIGS),
)
workflow_paths = WorkflowPaths(
    data_path=Path(PERISHABLE_GOODS_DATA_PATH),
    experiment_results_path=Path(VQR_EXPERIMENT_RESULTS_PATH),
    loss_history_path=Path(VQR_LOSS_HISTORY_PATH),
    model_dir=Path(PERISHABLE_VQR_MODEL_PATH),
)
print(f"Repository root: {repository_root}")
print(f"Dataset: {workflow_paths.data_path}")

## 5. Load dataset

In [ ]:
dataset = load_dataset(workflow_paths.data_path)
print(f"Loaded shape: {dataset.shape}")
display(dataset.head())

## 6. Validate dataset

Validation reports missing values, infinities, constants, duplicate rows, unsafe numeric conversions, and sample-size constraints. It raises instead of silently dropping problematic rows.

In [ ]:
dataset_report = validate_dataset(dataset, config=workflow_config)

## 7. Leakage review

`units_sold`, revenue/profit fields, waste/spoilage outcomes, and future demand/sales fields are never candidates. **`spoilage_risk` must remain excluded unless it is genuinely available before the forecast and is not calculated from future sales, waste, or spoilage outcomes.**

In [ ]:
leakage_report = review_target_leakage(
    dataset,
    include_spoilage_risk=workflow_config.include_spoilage_risk,
)

## 8. Prepare raw features and target

This step only copies the allow-listed raw inputs and target. It does not fit an encoder, selector, or scaler.

In [ ]:
raw_inputs = prepare_raw_inputs(
    dataset,
    include_spoilage_risk=workflow_config.include_spoilage_risk,
)
print("Raw input columns:", raw_inputs.columns.tolist())

## 9. Split train, validation, and test data

If `transaction_date` exists and parses cleanly, the split is chronological. Otherwise the workflow uses a deterministic random split and prints the forecasting limitation. Sampling happens only inside each already isolated subset.

In [ ]:
raw_split = split_raw_data(raw_inputs, workflow_config)
print("Split method:", raw_split.method)
print(
    "Sample sizes:",
    {"train": len(raw_split.train), "validation": len(raw_split.validation), "test": len(raw_split.test)},
)

## 10. Configure feature selection

Categorical vocabulary is learned from training data only when raw `category` or `region` columns are present. The current featured CSV is already encoded, so it is not encoded twice. Every experiment creates a fresh deterministic mutual-information selector and fits it only on training data.

In [ ]:
candidate_schema = fit_candidate_schema(
    raw_split.train,
    include_spoilage_risk=workflow_config.include_spoilage_risk,
    split_method=raw_split.method,
)
prepared_data = build_prepared_data(raw_split, candidate_schema)
print("Deterministic candidate order:")
print(prepared_data.X_train.columns.tolist())
print("Categorical encoding:", candidate_schema["categorical_encoding"])
print("Available candidate count:", prepared_data.X_train.shape[1])

## 11. Configure VQR experiments

The candidates use shallow functional `zz_feature_map`/`z_feature_map` and `real_amplitudes` circuits with `StatevectorEstimator`. No quantum hardware is configured.

In [ ]:
active_experiment_configs = effective_experiment_configs(workflow_config)
display(pd.DataFrame([config.__dict__ for config in active_experiment_configs]))

## 12. Train classical baseline

Ridge is fitted for each valid feature count using the same split, training-only selector, X scaler, and scaled target convention. Metrics are calculated after inverse transformation in original `units_sold` units.

In [ ]:
baseline_rows = run_classical_baselines(prepared_data, workflow_config)
display(pd.DataFrame(baseline_rows))

## 13. Define experiment runner

The imported runner creates a fresh selector, scalers, circuit, optimizer, estimator, and callback per experiment. It records failures and continues, clips negative predictions only after inverse target scaling, and releases model references between experiments.

In [ ]:
experiment_runner = run_vqr_experiments
print("Experiment runner ready. The next cell starts VQR fitting.")

## 14. Run VQR experiments

**Expensive cell:** review the configuration above before running. This is the first cell that fits a quantum model.

In [ ]:
vqr_rows, loss_rows = experiment_runner(prepared_data, workflow_config)

## 15. Compare validation results

The test set has not been used. The tables below contain validation metrics and training cost. MAPE excludes targets with absolute value at or below `1e-8`; SMAPE also protects its near-zero denominator. Neither percentage metric is used alone for selection.

In [ ]:
results, loss_history = save_experiment_tables(
    [*baseline_rows, *vqr_rows],
    loss_rows,
    workflow_paths,
)
display(results.sort_values(["model_name", "feature_count", "experiment_id"]))
plot_validation_metric(results, "r2_score", "R²")
plot_validation_metric(results, "mae", "MAE")
plot_training_duration(results)

## 16. Select winning VQR

Successful VQR rows are ranked by highest validation R², then lower MAE, lower RMSE, and finally lower training time within approximately equal metric groups.

In [ ]:
ranked_vqr = rank_successful_vqr(results)
display(ranked_vqr.head(10))
winning_row = ranked_vqr.iloc[0]
print("Winning validation experiment:", winning_row["experiment_id"])
print("Selected features:", winning_row["selected_features"])
plot_loss_curve(loss_history, str(winning_row["experiment_id"]))

## 17. Final test evaluation

The winning configuration is retrained from a fresh training-only selector/scalers/model fit, then evaluated once on the untouched test set. A Ridge model is also evaluated on the same selected features for context.

In [ ]:
final_result = fit_winner_and_evaluate_test(
    prepared_data,
    winning_row,
    random_state=workflow_config.random_state,
)
loss_rows.extend(final_result.loss_rows)
results, loss_history = save_experiment_tables(
    [*baseline_rows, *vqr_rows],
    loss_rows,
    workflow_paths,
)
print("Exact selected feature order:", final_result.matrices.selected_features)
print("VQR test metrics:", final_result.test_metrics)
print("Ridge test metrics:", final_result.baseline_test_metrics)
plot_loss_curve(loss_history, "final_winner_retrain")
plot_actual_vs_predicted(prepared_data.y_test, final_result.test_predictions)

## 18. Save model artifacts

The VQR is saved with Qiskit Machine Learning's dill persistence API. Scalers and selector use joblib; schema and metadata use JSON.

In [ ]:
artifact_paths = save_winning_artifacts(
    final_result,
    prepared_data,
    workflow_paths,
)
for artifact_name, artifact_path in artifact_paths.items():
    print(f"{artifact_name}: {artifact_path}")

# Stop using the in-memory fitted VQR before the deployment reload test.
final_result.model = None
gc.collect()

## 19. Reload and inference test

This deployment check reloads the VQR, selector, scalers, and schema from disk. It transforms one raw test row using only `transform` calls—never `fit` or `fit_transform`—then writes a numeric, non-negative sample prediction.

In [ ]:
reload_result = reload_and_test_inference(
    prepared_data.raw_test.iloc[[0]].copy(),
    artifact_paths,
)
assert isinstance(reload_result["predicted_units_sold"], float)
assert reload_result["predicted_units_sold"] >= 0.0
display(reload_result)

## 20. Files produced

In [ ]:
produced_files = [
    workflow_paths.experiment_results_path,
    workflow_paths.loss_history_path,
    artifact_paths["model"],
    artifact_paths["x_scaler"],
    artifact_paths["y_scaler"],
    artifact_paths["selector"],
    artifact_paths["feature_schema"],
    artifact_paths["metadata"],
    artifact_paths["test_prediction"],
]
for produced_file in produced_files:
    print(produced_file)
    if not Path(produced_file).is_file():
        raise FileNotFoundError(f"Expected output was not produced: {produced_file}")